# Mezcla de pesos de v5
Promedia los pesos de varios checkpoints del mismo linaje (v5 y v5b) y exporta cada mezcla. Sin entrenamiento.

In [ ]:
%pip install -q "ultralytics==8.4.161"
import shutil
from pathlib import Path
from ultralytics import YOLO
!rm -rf /content/tmp && git clone -q --depth 1 -b v5-tmp https://github.com/dportilla219/eggs-detector /content/tmp
import torch
import ultralytics.utils.export.litert as litert_mod
from ultralytics.utils.export.engine import _NormalizeCoords

_torch2litert_original = litert_mod.torch2litert

class _EntradaNHWC(torch.nn.Module):
    """Recibe [1, H, W, 3] (lo que entrega la cámara) y lo pasa al modelo como [1, 3, H, W]."""
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, x):
        return self.model(x.permute(0, 3, 1, 2))

class _CalibracionNHWC:
    """Pasa las imágenes de calibración INT8 a NHWC para que coincidan con la nueva entrada."""
    def __init__(self, loader):
        self.loader = loader
    def __iter__(self):
        for batch in self.loader:
            batch = dict(batch)
            batch['img'] = batch['img'].permute(0, 2, 3, 1)
            yield batch

def _torch2litert_nhwc(model, im, file, quantize, calibration_dataset, metadata, prefix):
    h, w = int(im.shape[2]), int(im.shape[3])
    model = _EntradaNHWC(_NormalizeCoords(model, h, w, 'detect', len((metadata or {}).get('names', {})), None))
    normalize_original = litert_mod._NormalizeCoords
    litert_mod._NormalizeCoords = lambda m, *a, **k: m  # ya normalizado arriba
    try:
        return _torch2litert_original(
            model, im.permute(0, 2, 3, 1).contiguous(), file, quantize,
            _CalibracionNHWC(calibration_dataset) if calibration_dataset is not None else None, metadata, prefix)
    finally:
        litert_mod._NormalizeCoords = normalize_original

litert_mod.torch2litert = _torch2litert_nhwc


T = Path('/content/tmp')
MEZCLAS = {'v5s2': ['v5b_best.pt', 'v5b_last.pt'], 'v5s3': ['v5_best.pt', 'v5b_best.pt', 'v5b_last.pt']}
out = Path('/content/eggs_v5_mezclas'); shutil.rmtree(out, ignore_errors=True); out.mkdir()
for nombre, pts in MEZCLAS.items():
    ms = [YOLO(str(T / p)) for p in pts]
    sd = [m.model.float().state_dict() for m in ms]
    avg = {k: (sum(s[k].float() for s in sd) / len(sd)) if sd[0][k].dtype.is_floating_point else sd[0][k] for k in sd[0]}
    ms[0].model.load_state_dict(avg)
    w = Path('/content/exp') / nombre; shutil.rmtree(w, ignore_errors=True); w.mkdir(parents=True)
    ms[0].save(str(w / f'eggs_{nombre}.pt'))
    f = Path(YOLO(str(w / f'eggs_{nombre}.pt')).export(format='litert', imgsz=640))
    shutil.copy2(f, out / f'eggs_{nombre}_fp32.tflite'); shutil.copy2(w / f'eggs_{nombre}.pt', out / f'eggs_{nombre}.pt')
    print(nombre, pts)
z = shutil.make_archive(str(out), 'zip', out)
from google.colab import files
files.download(z)
